# Scratchpad for utils.py functions

## Load CSV

In [2]:
import os
import pandas as pd

csv_path = 'sample_data/ponyodat_new.csv' 

if os.path.exists(csv_path):
    print(f"Loading data from: {csv_path}")
    
    df = pd.read_csv(csv_path, nrows=50) # Reads only first 50 rows for quick inspection. Remove nrows for full dataset.
    
    print(f"Loaded {len(df)} rows.")
    print(f"Columns: {list(df.columns)}")
    display(df.head())
else:
    raise FileNotFoundError(f"File not found at '{csv_path}'. Please check the path.")   

Loading data from: sample_data/ponyodat_new.csv
Loaded 50 rows.
Columns: ['Timestamp', 'q1']


,Timestamp,q1
0,2026-04-13 02:39:50,"{""ts"":""2026-04-13T06:39:49Z"",""pm"":{""PM1_0_ATM""..."
1,2026-04-13 02:39:55,"{""ts"":""2026-04-13T06:39:54Z"",""dht"":{""humidity_..."
2,2026-04-13 02:40:00,"{""ts"":""2026-04-13T06:39:59Z"",""pm"":{""PM1_0_ATM""..."
3,2026-04-13 02:40:05,"{""ts"":""2026-04-13T06:40:04Z"",""pm"":{""PM1_0_ATM""..."
4,2026-04-13 02:40:10,"{""ts"":""2026-04-13T06:40:09Z"",""dht"":{""humidity_..."


## JSON-Blob Parser

In [3]:
# For any input with JSON-blob-in-CSV data

import re

import json
import pandas as pd 

def safe_json_loads(s):
    """Safely parse JSON string; returns empty dict on failure."""
    try:
        return json.loads(s) if pd.notna(s) else {}
    except (json.JSONDecodeError, TypeError):
        return {}

def flatten_dict(d, parent_key='', sep='.'):
    """Recursively flatten nested dictionaries."""
    items = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.update(flatten_dict(v, new_key, sep=sep))
        else:
            items[new_key] = v
    return items


def detect_json_col(df, sample_size=100):
    for col in df.columns:
        if df[col].dtype not in ['object', 'string']:
            continue
        sample = df[col].dropna().head(sample_size)
        if len(sample) == 0:
            continue
        json_count = 0
        for val in sample:
            if not isinstance(val, str):
                val = str(val)
            val = val.strip()
            if len(val) >= 2:
                if (val.startswith('"') and val.endswith('"')) or \
                (val.startswith("'") and val.endswith("'")):
                    val = val[1:-1].strip()
            if val.startswith('{') or val.startswith('['):
                try:
                    json.loads(val)
                    json_count += 1
                except:
                    (json.JSONDecodeError, TypeError)
                    pass
        if json_count > (len(sample) * 0.1): # threshold dropped to 10% for better sensitivity to any col that could be JSON-like
            return col
    return None

def parse_jsonblob_csv(df, json_col=None, keep_original_cols=None):
    """
    Process DataFrame with a JSON blob column: auto-detects the JSON column,
    flattens it, and keeps all other original columns.
    """
    df = df.copy()

    # 1. Auto-detect JSON column
    if json_col is None:
        json_col = detect_json_col(df)
        if json_col is None:
            raise ValueError("No JSON column detected. Please specify 'json_col' manually.")
        print(f"Auto-detected JSON column: '{json_col}'")

    # 2. Validate JSON column exists
    if json_col not in df.columns:
        raise KeyError(f"Column '{json_col}' not found. Available: {list(df.columns)}")

    # 3. Set default for keeping original columns
    if keep_original_cols is None:
        # Keep all columns except the JSON column being flattened
        keep_original_cols = [col for col in df.columns if col != json_col]
    else:
        # Validate user-specified columns exist
        missing = [c for c in keep_original_cols if c not in df.columns]
        if missing:
            raise KeyError(f"Columns {missing} not found in DataFrame.")

    # 4. Parse JSON
    df[json_col] = df[json_col].apply(safe_json_loads)

    # 5. Flatten JSON and keep original columns
    # Create a list of DataFrames to concatenate
    dfs_to_concat = []

    # 6. Add the original non-JSON columns
    if keep_original_cols:
        dfs_to_concat.append(df[keep_original_cols].reset_index(drop=True))

    # 7. Add the flattened JSON columns
    flattened_data = []
    for _, row in df.iterrows():
        flattened_data.append(flatten_dict(row[json_col]))
    flattened_df = pd.DataFrame(flattened_data)
    dfs_to_concat.append(flattened_df)

    # 8. Combine everything
    final_df = pd.concat(dfs_to_concat, axis=1)
    return final_df   

In [4]:
try:
    df_processed = parse_jsonblob_csv(df)
    
    print("\nSuccess! Data flattened:")
    print(f"New shape: {df_processed.shape}")
    display(df_processed.head())
    
    print(f"\nColumns generated: {list(df_processed.columns)}")
    
except Exception as e:
    print(f"\nFailed: {e}")
    import traceback
    traceback.print_exc()   

Auto-detected JSON column: 'q1'

Success! Data flattened:
New shape: (50, 38)


,Timestamp,ts,pm.PM1_0_ATM,pm.PM2_5_ATM,pm.PM10_0_ATM,gps.lat,gps.lon,gps.alt_m,gps.sats,gps.fix_quality,...,power.freq_capped_occurred,power.throttled_occurred,power.soft_temp_occurred,power.vcore_v,power.vsdram_c_v,power.vsdram_i_v,power.vsdram_p_v,errors,dht.humidity_pct,dht.temp_c
0,2026-04-13 02:39:50,2026-04-13T06:39:49Z,3,7,8,0.000000,0.000000,0,0,,...,False,False,False,1.275,1.25,1.25,1.275,[dht:missing some readings - high level durati...,NaN,NaN
1,2026-04-13 02:39:55,2026-04-13T06:39:54Z,3,9,9,43.705961,-79.397914,0,6,,...,False,False,False,1.275,1.25,1.25,1.275,NaN,66.9,16.4
2,2026-04-13 02:40:00,2026-04-13T06:39:59Z,3,9,9,43.705961,-79.397914,0,6,,...,False,False,False,1.275,1.20,1.20,1.200,[dht:missing some readings - low level duratio...,NaN,NaN
3,2026-04-13 02:40:05,2026-04-13T06:40:04Z,4,11,11,43.705961,-79.397914,0,6,,...,False,False,False,1.275,1.20,1.20,1.200,[dht:missing some readings - high level durati...,NaN,NaN
4,2026-04-13 02:40:10,2026-04-13T06:40:09Z,5,12,12,43.705961,-79.397914,0,5,,...,False,False,False,1.275,1.20,1.20,1.200,NaN,66.2,16.5



Columns generated: ['Timestamp', 'ts', 'pm.PM1_0_ATM', 'pm.PM2_5_ATM', 'pm.PM10_0_ATM', 'gps.lat', 'gps.lon', 'gps.alt_m', 'gps.sats', 'gps.fix_quality', 'gps.has_fix', 'net.local_ip', 'net.public_ip', 'sys.cpu_pct', 'sys.cpu_temp_c', 'sys.load1', 'sys.load5', 'sys.load15', 'sys.mem_total_mb', 'sys.mem_avail_mb', 'sys.mem_used_mb', 'sys.uptime_s', 'power.raw_hex', 'power.undervolt_now', 'power.freq_capped_now', 'power.throttled_now', 'power.soft_temp_now', 'power.undervolt_occurred', 'power.freq_capped_occurred', 'power.throttled_occurred', 'power.soft_temp_occurred', 'power.vcore_v', 'power.vsdram_c_v', 'power.vsdram_i_v', 'power.vsdram_p_v', 'errors', 'dht.humidity_pct', 'dht.temp_c']


In [6]:
import sys
sys.path.insert(0, '/home/yul/Projects/Repos/dataDashboard')

import importlib
import src.utils as utils
importlib.reload(utils)

from src.utils import add_timezone_col, build_gis_df

gps_keywords = ['lat', 'lon', 'lng', 'longitude', 'latitude', 'gps', 'coord']
gps_cols = [c for c in df_processed.columns if any(kw in c.lower() for kw in gps_keywords)]
print(df_processed[gps_cols].dtypes)
print(df_processed[gps_cols].head())

df_with_tz = add_timezone_col(df_processed)
print(df_with_tz[['datetime', 'latitude', 'longitude', 'timezone']].head())

gps.lat            float64
gps.lon            float64
gps.alt_m            int64
gps.sats             int64
gps.fix_quality        str
gps.has_fix           bool
dtype: object
     gps.lat    gps.lon  gps.alt_m  gps.sats gps.fix_quality  gps.has_fix
0   0.000000   0.000000          0         0                        False
1  43.705961 -79.397914          0         6                         True
2  43.705961 -79.397914          0         6                         True
3  43.705961 -79.397914          0         6                         True
4  43.705961 -79.397914          0         5                         True
Detected datetime column: 'ts' (score: 7.00) → renamed to 'datetime'
Detected lat column: 'gps.lat' → renamed to 'latitude'
Detected lon column: 'gps.lon' → renamed to 'longitude'
                   datetime   latitude  longitude         timezone
0 2026-04-13 06:39:49+00:00   0.000000   0.000000             <NA>
1 2026-04-13 06:39:54+00:00  43.705961 -79.397914  America/Toronto

## UTC Column Detector

There may be columns like 'timestamp' or 'ts' or 'date' or 'time', and it might have an actual UTC ISO 8601 format in it or something close. There also may be rows with more than one time/date column in it. Sometimes it has the local time or just UTC. 

So, the detector should just identify if there is more than one date/time/datetime column, and then choose the more legimate of the two to set it as the first column in the data frame and rename it to just 'datetime' for standardization.

In [ ]:
# For any input with UTC datetime + lat/lon columns

import re

import json
import pandas as pd 

from tzfpy import get_tz

def detect_utc_col(df, sample_size=50):
    """
    Detect all datetime-like columns, rank them by legitimacy,
    rename the best one to 'datetime', and move it to the first column.
    Returns the updated DataFrame and the detected column name.
    """
    utc_pattern = re.compile(
        r'\d{4}-\d{2}-\d{2}[ T]\d{2}:\d{2}:\d{2}'
    )
    tz_aware_pattern = re.compile(
        r'(\+\d{2}:\d{2}|Z|UTC)$'
    )
    name_keywords = ['time', 'date', 'datetime', 'timestamp', 'ts']

    candidates = {}

    for col in df.columns:
        score = 0
        col_lower = col.lower()

        if any(kw in col_lower for kw in name_keywords):
            score += 1

        if df[col].dtype not in ['object', 'string']:
            if pd.api.types.is_datetime64_any_dtype(df[col]):
                score += 5
                score += df[col].notna().mean()
                candidates[col] = score
            continue

        sample = df[col].dropna().head(sample_size)
        if len(sample) == 0:
            continue

        iso_matches = sum(1 for v in sample if utc_pattern.match(str(v).strip()))
        tz_matches  = sum(1 for v in sample if tz_aware_pattern.search(str(v).strip()))
        fill_rate   = df[col].notna().mean()

        if iso_matches == 0:
            continue

        score += (iso_matches / len(sample)) * 3
        score += (tz_matches  / len(sample)) * 2
        score += fill_rate

        candidates[col] = score

    if not candidates:
        return df, None

    best_col = max(candidates, key=candidates.get)

    df = df.rename(columns={best_col: 'datetime'})
    cols = ['datetime'] + [c for c in df.columns if c != 'datetime']
    df = df[cols]

    print(f"Detected datetime column: '{best_col}' (score: {candidates[best_col]:.2f}) → renamed to 'datetime'")

    return df, best_col

# Example: df, col = detect_utc_col(df)                         # standalone — detects, renames, reorders
# Example: add_timezone_col(df)                            # auto-detection
# Example: add_timezone_col(df, datetime_col='timestamp')  # manual override

In [ ]:
try:
    # detect_utc_col returns the DataFrame AND the detected column name
    df_with_datetime, detected_col = detect_utc_col(df_processed)
    display(df_with_datetime.head())
    print(type(df_processed))
    print(df_processed.shape)
    print(f"\nColumns generated: {list(df_with_datetime.columns)}")
    if detected_col:
        print(f"Detected and renamed: '{detected_col}' -> 'datetime'")
    else:
        print("No datetime column found.")
        
except Exception as e:
    print(f"\nFailed: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
try:
    df_with_tz = add_timezone_col(df_with_datetime)
    display(df_with_tz.head())
    print(f"\nColumns generated: {list(df_with_tz.columns)}")
    print(f"Timezone sample:\n{df_with_tz['tz_string'].value_counts()}")

except ValueError as e:
    print(f"Detection failed: {e}")

except KeyError as e:
    print(f"Column not found: {e}")

except Exception as e:
    print(f"Failed: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
def detect_latlon_cols(df, sample_size=50):
    """
    Detect latitude and longitude columns by name keywords and content range.
    Renames detected columns to 'lat' and 'lon' for standardization.
    Returns updated DataFrame and (lat_col, lon_col) or (None, None) if not found.
    """
    lat_keywords = ['lat', 'latitude', 'y', 'yloc', 'northing']
    lon_keywords = ['lon', 'lng', 'longitude', 'x', 'xloc', 'easting']

    lat_candidates = {}
    lon_candidates = {}

    for col in df.columns:
        col_lower = col.lower()

        if not pd.api.types.is_numeric_dtype(df[col]):
            continue

        sample = df[col].dropna().head(sample_size)
        if len(sample) == 0:
            continue

        col_min, col_max = sample.min(), sample.max()

        if any(kw in col_lower for kw in lat_keywords):
            score = 1
            if -90 <= col_min and col_max <= 90:
                score += 3
            lat_candidates[col] = score

        if any(kw in col_lower for kw in lon_keywords):
            score = 1
            if -180 <= col_min and col_max <= 180:
                score += 3
            lon_candidates[col] = score

    lat_col = max(lat_candidates, key=lat_candidates.get) if lat_candidates else None
    lon_col = max(lon_candidates, key=lon_candidates.get) if lon_candidates else None

    # Rename to standardized names
    rename_map = {}
    if lat_col and lat_col != 'lat':
        rename_map[lat_col] = 'latitude'
        print(f"Detected lat column: '{lat_col}' → renamed to 'latitude'")
    elif lat_col:
        print(f"Detected lat column: '{lat_col}'")

    if lon_col and lon_col != 'lon':
        rename_map[lon_col] = 'longitude'
        print(f"Detected lon column: '{lon_col}' → renamed to 'longitude'")
    elif lon_col:
        print(f"Detected lon column: '{lon_col}'")

    if not lat_col or not lon_col:
        print("Warning: Could not detect lat/lon columns — timezone will default to UTC.")

    df = df.rename(columns=rename_map)

    return df, 'latitude' if lat_col else None, 'longitude' if lon_col else None

def add_timezone_col(df, datetime_col=None, lat_col=None, lon_col=None):
    """
    Add timezone metadata but keep datetime in UTC.
    Auto-detects and standardizes UTC datetime, lat, and lon columns if not specified.
    Returns DataFrame with columns reordered: datetime, lon, lat, timezone first.
    """
    df = df.copy()

    # 1. Auto-detect and standardize datetime column
    if datetime_col is None:
        df, datetime_col = detect_utc_col(df)

    # 2. Auto-detect and standardize lat/lon columns
    if lat_col is None or lon_col is None:
        df, lat_col, lon_col = detect_latlon_cols(df)

    # 3. Ensure datetime is UTC-aware
    df['datetime'] = pd.to_datetime(df['datetime'], utc=True)

    # 4. Identify first row of valid coordinates
    if lat_col and lon_col:
        valid_coords = (
            df[lat_col].notna() & df[lon_col].notna() &
            (df[lat_col] != 0.0) & (df[lon_col] != 0.0) # Excludes zero coordinates which are often placeholders for missing data
)
    else:
        valid_coords = pd.Series(False, index=df.index)

    # 5. Add timezone
    if valid_coords.any():
        df.loc[valid_coords, 'timezone'] = df.loc[valid_coords].apply(
            lambda row: get_tz(lng=row[lon_col], lat=row[lat_col]),
            axis=1
        )
        df['timezone'] = df['timezone'].bfill().ffill() # Backwards fills first, then forward fills to propagate the first valid timezone to any rows missing coordinates
    else:
        df['timezone'] = 'UTC'

    # 6. Reorder: datetime, lon, lat, timezone first — then all remaining columns
    priority_cols = [c for c in ['datetime', 'longitude', 'latitude', 'timezone'] if c in df.columns]
    remaining_cols = [c for c in df.columns if c not in priority_cols]
    df = df[priority_cols + remaining_cols]

    return df

# Example: add_timezone_col(df)                                                         # auto-detection
# Example: add_timezone_col(df, lat_col='gps.lat', lon_col='gps.lon')                   # manual lat/lon
# Example: add_timezone_col(df, datetime_col='timestamp', lat_col='lat', lon_col='lon') # fully manual

In [ ]:
try:
    df_with_tz = add_timezone_col(df_with_datetime)
    display(df_with_tz.head())
    print(f"\nColumns generated: {list(df_with_tz.columns)}")
    print(f"\nTimezone sample:\n{df_with_tz['tz_string'].value_counts()}")
    print(f"\nDatetime sample:\n{df_with_tz['datetime'].head()}")

except ValueError as e:
    print(f"Detection failed: {e}")

except KeyError as e:
    print(f"Column not found: {e}")

except Exception as e:
    print(f"Failed: {e}")
    import traceback
    traceback.print_exc()